In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.model_selection import train_test_split
import joblib
import glob  # for file matching

# List all CSV files you want to load (adjust path/pattern)
file_paths = glob.glob(r"C:\Users\debas\AIRTRAJ\dataset-research\Data Preprocessing\Oslo\processed_data\*.csv")

# Load and concatenate all files
df_list = []
for file in file_paths:
    temp_df = pd.read_csv(file)
    df_list.append(temp_df)

# Combine all data into a single DataFrame
df = pd.concat(df_list, ignore_index=True)

# Use only these 4 features for input & prediction
features = ['lat', 'lon', 'alt', 'time']
data = df[features]

# Scale all features
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# Save the scaler for later use
joblib.dump(scaler, "coord_scaler.pkl")

# Create sequences function (unchanged)
def create_sequences(data, seq_length=10):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])  # Next timestep
    return np.array(X), np.array(y)

SEQ_LENGTH = 10
X, y = create_sequences(data_scaled, SEQ_LENGTH)

# Split data (no shuffle to preserve time order)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Build LSTM model
model = Sequential([
    LSTM(64, input_shape=(SEQ_LENGTH, len(features)), activation='tanh'),
    Dense(4)  # Output: lat, lon, alt, time
])

model.compile(optimizer='adam', loss='mse')

# Train
model.fit(X_train, y_train, epochs=3, batch_size=32, validation_split=0.1)

# Save model using new .keras format
model.save("coord_predictor.keras")


c:\Users\debas\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/3
93139/93139 ━━━━━━━━━━━━━━━━━━━━ 823s 9ms/step - loss: 0.0103 - val_loss: 0.0130
Epoch 2/3
93139/93139 ━━━━━━━━━━━━━━━━━━━━ 594s 6ms/step - loss: 0.0094 - val_loss: 0.0112
Epoch 3/3
93139/93139 ━━━━━━━━━━━━━━━━━━━━ 577s 6ms/step - loss: 0.0090 - val_loss: 0.0103


In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
import joblib
import json

def predict_from_csv(csv_path):
    # ----------------------
    # Load model & scaler
    # ----------------------
    MODEL_PATH = r"C:\Users\debas\AIRTRAJ\dataset-research\Data Annotation UI\coord_predictor_delta.keras"
    SCALER_PATH = r"C:\Users\debas\AIRTRAJ\dataset-research\Data Annotation UI\coord_scaler_delta.pkl"

    model = load_model(MODEL_PATH, compile=False)
    scaler = joblib.load(SCALER_PATH)

    # ----------------------
    # Load data & compute deltas
    # ----------------------
    df = pd.read_csv(csv_path)
    df['time_delta'] = df['time'].diff().fillna(0)  # first delta = 0
    features = ['lat', 'lon', 'alt', 'time_delta']
    data = df[features]
    data_scaled = scaler.transform(data)

    # ----------------------
    # Sequence creation
    # ----------------------
    SEQ_LENGTH = 10
    def create_sequences(data, seq_length=SEQ_LENGTH):
        X, y = [], []
        for i in range(len(data) - seq_length):
            X.append(data[i:i+seq_length])
            y.append(data[i+seq_length])
        return np.array(X), np.array(y)

    X, y_true_scaled = create_sequences(data_scaled, SEQ_LENGTH)

    # ----------------------
    # Prediction
    # ----------------------
    y_pred_scaled = model.predict(X)
    y_pred = scaler.inverse_transform(y_pred_scaled)
    y_true = scaler.inverse_transform(y_true_scaled)

    # Extract true and predicted columns
    true_lat, true_lon, true_alt, true_delta_time = y_true.T
    pred_lat, pred_lon, pred_alt, pred_delta_time = y_pred.T

    # ----------------------
    # Reconstruct absolute predicted time
    # ----------------------
    pred_delta_time = np.maximum(pred_delta_time, 0)  # avoid negative jumps
    pred_time = np.cumsum(pred_delta_time) + df['time'].iloc[0]
    true_time = np.cumsum(true_delta_time) + df['time'].iloc[0]

    # ----------------------
    # Accuracy calculations
    # ----------------------
    def haversine(lat1, lon1, lat2, lon2):
        R = 6371000
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        dphi = np.radians(lat2 - lat1)
        dlambda = np.radians(lon2 - lon1)
        a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
        return 2 * R * np.arcsin(np.sqrt(a))

    horizontal_error = haversine(true_lat, true_lon, pred_lat, pred_lon)
    alt_error = np.abs(true_alt - pred_alt)
    error_3d = np.sqrt(horizontal_error**2 + alt_error**2)

    segment_distances = haversine(true_lat[:-1], true_lon[:-1], true_lat[1:], true_lon[1:])
    total_flight_length_m = np.sum(segment_distances)
    avg_3d_error_m = np.mean(error_3d)
    percentage_error = (avg_3d_error_m / total_flight_length_m) * 100

    # ----------------------
    # Save results
    # ----------------------
    results = {
        "sample_predictions": {
            "true_latitude": [round(x, 6) for x in true_lat[:5].tolist()],
            "true_longitude": [round(x, 6) for x in true_lon[:5].tolist()],
            "true_altitude_m": [round(x, 2) for x in true_alt[:5].tolist()],
            "predicted_latitude": [round(x, 6) for x in pred_lat[:5].tolist()],
            "predicted_longitude": [round(x, 6) for x in pred_lon[:5].tolist()],
            "predicted_altitude_m": [round(x, 2) for x in pred_alt[:5].tolist()],
            "predicted_time": [round(x, 2) for x in pred_time[:5].tolist()]
        },
        "metrics": {
            "Total Flight Path Length (km)": round(total_flight_length_m / 1000, 3),
            "Average Horizontal Error (m)": round(float(np.mean(horizontal_error)), 2),
            "Average Altitude Error (m)": round(float(np.mean(alt_error)), 2),
            "Average 3D Error (m)": round(float(avg_3d_error_m), 2),
            "Prediction Error (% of Path)": round(float(percentage_error), 4),
        }
    }

    with open("prediction_results.json", "w") as f:
        json.dump(results, f, indent=2)

    # ----------------------
    # Plot results
    # ----------------------
    fig = plt.figure(figsize=(14, 6))

    # 3D Trajectory
    ax = fig.add_subplot(121, projection='3d')
    ax.plot(true_lat, true_lon, true_alt, label='Actual', alpha=0.7)
    ax.plot(pred_lat, pred_lon, pred_alt, label='Predicted', linestyle='--', alpha=0.7)
    ax.set_xlabel('Latitude')
    ax.set_ylabel('Longitude')
    ax.set_zlabel('Altitude')
    ax.set_title('3D Trajectory: Actual vs Predicted')
    ax.legend()

    # Time plot (reconstructed)
    ax2 = fig.add_subplot(122)
    seq_steps = np.arange(len(true_time))
    ax2.plot(seq_steps, true_time, label='Actual Time', alpha=0.7)
    ax2.plot(seq_steps, pred_time, label='Predicted Time', linestyle='--', alpha=0.7)
    ax2.set_title('Time Prediction')
    ax2.set_xlabel('Sequence Step')
    ax2.set_ylabel('Time')
    ax2.legend()

    plt.tight_layout()
    
    # Save and show
    static_dir = os.path.join(os.path.dirname(__file__), "static", "plots")
    os.makedirs(static_dir, exist_ok=True)
    plt.savefig(os.path.join(static_dir, "prediction_plot.png"))
    plt.show()

    return results
